# Foundations A1: Tensors and Shapes

Companion notebook for [Foundations A1 of the interactive course](https://bc-sparknet.github.io/make-an-llm-2/#/chapter/tensors). The code cells follow the chapter's examples in order; the explanations are on the site.

*This notebook is generated from the chapter source. To change it, edit the chapter and run `npm run notebooks`.*

In [ ]:
%pip install -q torch tiktoken

## What a tensor is

In [ ]:
import torch

loss = torch.tensor(3.7)                       # 0-D: a scalar
emb = torch.tensor([0.2, -1.3, 0.8, 0.5])      # 1-D: a vector
sent = torch.tensor([[0.2, -1.3, 0.8, 0.5],    # 2-D: a matrix, one row per token
                     [0.9, 0.1, -0.4, 0.3],
                     [-0.7, 0.6, 0.2, -1.1]])

for t in (loss, emb, sent):
    print(t.ndim, t.shape)
# 0 torch.Size([])
# 1 torch.Size([4])
# 2 torch.Size([3, 4])

In [ ]:
x = torch.rand(2, 3, 4)        # 3-D: 2 sentences × 3 tokens × 4 features
q = torch.rand(2, 12, 3, 64)   # 4-D: 2 sentences × 12 heads × 3 tokens × 64 features

print(x.ndim, x.shape)         # 3 torch.Size([2, 3, 4])
print(q.ndim, q.shape)         # 4 torch.Size([2, 12, 3, 64])
print(x.shape[0], x.shape[-1]) # 2 4   (a shape is indexable, like a tuple)
print(x.numel())               # 24    (total number of values: 2 × 3 × 4)

### Integers and floats

In [ ]:
ids = torch.tensor([15496, 11, 314, 716])   # "Hello, I am" as GPT-2 token IDs

print(ids.dtype)                  # torch.int64
print(emb.dtype)                  # torch.float32
print(ids.float().dtype)          # torch.float32   .float() converts a copy

## Reading a shape

In [ ]:
x = torch.tensor([
    [[ 0.1,  0.5, -0.3,  0.8],    # sentence 0, token 0: "the"
     [ 0.7, -0.2,  0.4,  0.0],    # sentence 0, token 1: "cat"
     [-0.6,  0.9,  0.2,  0.4]],   # sentence 0, token 2: "sat"
    [[ 0.3,  0.3, -0.8,  0.1],    # sentence 1, token 0: "a"
     [-0.4,  0.6,  0.5, -0.9],    # sentence 1, token 1: "dog"
     [ 0.2, -0.1,  0.7,  0.6]],   # sentence 1, token 2: "ran"
])

print(x.shape)        # torch.Size([2, 3, 4])
print(len(x))         # 2   len() counts the outermost dimension...
print(len(x[0]))      # 3   ...one level in: tokens in sentence 0
print(len(x[0][0]))   # 4   ...and in again: features of one token

In [ ]:
batch_size, num_tokens, emb_dim = x.shape
print(batch_size, num_tokens, emb_dim)   # 2 3 4

### Four dimensions: a grid of matrices

In [ ]:
batch_size, num_heads, num_tokens, head_dim = 8, 12, 256, 64
q = torch.rand(batch_size, num_heads, num_tokens, head_dim)

print(q.shape)         # torch.Size([8, 12, 256, 64])
print(q[0].shape)      # torch.Size([12, 256, 64])   all 12 heads for sentence 0
print(q[0, 3].shape)   # torch.Size([256, 64])       one matrix: sentence 0, head 3

## Indexing and slicing

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)   # 2 sentences × 3 tokens × 4 features
print(x)
# tensor([[[ 0,  1,  2,  3],
#          [ 4,  5,  6,  7],
#          [ 8,  9, 10, 11]],
#
#         [[12, 13, 14, 15],
#          [16, 17, 18, 19],
#          [20, 21, 22, 23]]])

In [ ]:
print(x[0].shape)          # torch.Size([3, 4])   sentence 0: a matrix
print(x[1, 2])             # tensor([20, 21, 22, 23])   sentence 1, token 2
print(x[:, -1, :])         # the last token of every sentence
# tensor([[ 8,  9, 10, 11],
#         [20, 21, 22, 23]])
print(x[:, :, 0])          # feature 0 of every token
# tensor([[ 0,  4,  8],
#         [12, 16, 20]])
print(x[0, 1:3].shape)     # torch.Size([2, 4])   tokens 1 and 2 of sentence 0
print(x[..., -1])          # same as x[:, :, -1]
# tensor([[ 3,  7, 11],
#         [15, 19, 23]])
print(x[1, 2, 3])          # tensor(23)   one index per dim: a 0-D tensor
print(x[1, 2, 3].item())   # 23           .item() turns a 0-D tensor into a Python number

In [ ]:
print(x[:, -1, :].shape)   # torch.Size([2, 4])
print(x[:, -1:, :].shape)  # torch.Size([2, 1, 4])

### The one you'll use most: `[:, -1, :]`

In [ ]:
h = torch.rand(8, 256, 768)    # [batch, tokens, emb_dim]: 8 sequences of 256 tokens
last = h[:, -1, :]             # [batch, emb_dim]: the final token of each sequence
print(last.shape)              # torch.Size([8, 768])

## Creating tensors

In [ ]:
print(torch.zeros(2, 3))         # all zeros, shape [2, 3]
# tensor([[0., 0., 0.],
#         [0., 0., 0.]])
print(torch.ones(3))             # tensor([1., 1., 1.])
print(torch.arange(5))           # tensor([0, 1, 2, 3, 4])   like range(5)
print(torch.arange(2, 10, 3))    # tensor([2, 5, 8])         start, stop, step

In [ ]:
torch.manual_seed(123)
print(torch.rand(2, 3))
# tensor([[0.2961, 0.5166, 0.2517],
#         [0.6886, 0.0740, 0.8665]])

torch.manual_seed(123)
print(torch.rand(2, 3))          # exactly the same numbers again

### Adding and removing size-1 dimensions

In [ ]:
ids = torch.tensor([15496, 11, 314, 716])   # "Hello, I am"
print(ids.shape)                 # torch.Size([4])

batch = ids.unsqueeze(0)         # add a batch dimension at the front
print(batch.shape)               # torch.Size([1, 4])
print(batch)                     # tensor([[15496,    11,   314,   716]])

print(ids.unsqueeze(1).shape)    # torch.Size([4, 1])   new dim at the end instead
print(ids.unsqueeze(-1).shape)   # torch.Size([4, 1])   -1 means "at the end"
print(batch.squeeze(0).shape)    # torch.Size([4])      and back again

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0)   # [tokens] -> [1, tokens]

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())     # [1, tokens] -> [tokens]

batch = text_to_token_ids("Hello, I am", tokenizer)
print(batch.shape)                          # torch.Size([1, 4])
print(token_ids_to_text(batch, tokenizer))  # Hello, I am

In [ ]:
one = torch.rand(1, 1, 768)     # a batch of 1 sequence, 1 token long
print(one.squeeze().shape)      # torch.Size([768])      both size-1 dims gone!
print(one.squeeze(0).shape)     # torch.Size([1, 768])   only the batch dim

## Devices: CPU and GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)        # cuda on a GPU runtime, cpu otherwise

x = torch.rand(2, 3).to(device)
print(x.device)      # cuda:0 on a GPU, cpu otherwise
print(x.shape)       # torch.Size([2, 3])   moving never changes the shape

## Summary